# BT4012 Kaggle 2026 — Illicit Bitcoin Transaction Detection

**Task:** predict the probability that each test transaction is illicit. **Metric:** ROC AUC.

**Decisions so far**
- **Validation (A1):** three expanding-window forward folds, because test time steps (36–49) all come after training (1–35).
- **Stage 1 model choice (B3):** Random Forest vs default LightGBM on the raw features; LightGBM won (public LB 0.94664).
- **Selection rule (C1):** after stage 1, fold F1 (validate on steps 21–27) matched the public LB (0.9451 vs 0.9466) while F2/F3 were over-optimistic (~0.996).
  Experiments are therefore ranked by **F1 AUC**, with mean fold AUC as tie-break.
- **Stage 2 (D1):** graph features from the transaction edge list.

*AI use: Claude (Claude Code) was used to write boilerplate code for this notebook; validation, feature and model choices were made by me.*

In [1]:
import time
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

SEED = 42
DATA = Path('.')                 # folder holding train.csv, test.csv, txs_edgelist.csv, sample_submission.csv
SUB_DIR = Path('submissions'); SUB_DIR.mkdir(exist_ok=True)
LOG = Path('experiment_log.csv')
np.random.seed(SEED)

## 1. Load data

- `label` is blank for unknown transactions. Unknown is **not** licit, so those rows are excluded from supervised training and evaluation.
- Features are read as float32 to halve memory.

In [2]:
feat_cols = [f'feat_{i}' for i in range(1, 166)]
dtypes = {c: 'float32' for c in feat_cols}

train = pd.read_csv(DATA / 'train.csv', dtype=dtypes)
test = pd.read_csv(DATA / 'test.csv', dtype=dtypes)
edges = pd.read_csv(DATA / 'txs_edgelist.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')

lab = train[train['label'].notna()].copy()
lab['label'] = lab['label'].astype(int)

print('train', train.shape, '| labelled', lab.shape, '| test', test.shape, '| edges', edges.shape)
print('illicit rate (labelled):', round(lab['label'].mean(), 4))
print('train steps', train.time_step.min(), '-', train.time_step.max(), '| test steps', test.time_step.min(), '-', test.time_step.max())
assert (test['index'].values == sample['index'].values).all(), 'test order differs from sample submission'

train (141772, 168) | labelled (31235, 168) | test (15329, 168) | edges (234355, 2)
illicit rate (labelled): 0.1167
train steps 1 - 35 | test steps 36 - 49


## 2. Validation design (A1): forward-in-time folds

Each fold trains only on earlier steps and validates on later ones, mimicking the train → test gap.
`time_step` is used only for splitting, never as a model feature (test values 36–49 never occur in training).

In [3]:
FOLDS = {
    'F1': (range(1, 21), range(21, 28)),
    'F2': (range(1, 28), range(28, 36)),
    'F3': (range(1, 25), range(25, 36)),   # long horizon, closest to the 14-step test window
}
for name, (tr_steps, va_steps) in FOLDS.items():
    va = lab[lab.time_step.isin(va_steps)]
    print(f'{name}: train {len(lab[lab.time_step.isin(tr_steps)]):>6} rows | valid {len(va):>5} rows, {va.label.sum()} illicit')

F1: train  18889 rows | valid  6034 rows, 686 illicit
F2: train  24923 rows | valid  6312 rows, 1187 illicit
F3: train  23606 rows | valid  7629 rows, 1425 illicit


## 3. Graph structure checks

An edge `txId1 → txId2` means an output of `txId1` is spent by `txId2`. Checks that shape the feature design:
- every edge stays within one time step, and there are no train ↔ test edges, so each step is its own graph;
- the graph is acyclic (funds only flow forward);
- **in the test period, unlabelled transactions appear in the edge list but have no feature rows**, whereas in training they do.
  Any feature built from *neighbours' features* must therefore only use neighbours that would also be visible in the test period (labelled train rows and test rows).

In [4]:
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))

step = pd.concat([train[['txId', 'time_step']], test[['txId', 'time_step']]]).set_index('txId')['time_step']
e_step = edges.assign(s1=edges.txId1.map(step), s2=edges.txId2.map(step)).dropna()
train_ids, test_ids = set(train.txId), set(test.txId)

print('directed acyclic graph:', nx.is_directed_acyclic_graph(G))
print('edges crossing time steps:', int((e_step.s1 != e_step.s2).sum()))
print('train <-> test edges:', int((edges.txId1.isin(train_ids) & edges.txId2.isin(test_ids)).sum()
                                   + (edges.txId1.isin(test_ids) & edges.txId2.isin(train_ids)).sum()))
print('edge-list nodes without a feature row:', G.number_of_nodes() - len(set(G.nodes) & (train_ids | test_ids)))

directed acyclic graph: True
edges crossing time steps: 0
train <-> test edges: 0
edge-list nodes without a feature row: 46668


## 4. Feature engineering (D1)

### 4a. Structural features (full edge list)
Computed from the full edge list, which covers every transaction in both periods, so they are built identically for train and test.

| Feature | Meaning |
| --- | --- |
| `in_deg`, `out_deg`, `deg_ratio` | observed funding transactions in / spending transactions out; fan-out vs fan-in |
| `is_source`, `is_sink` | no observed inputs / outputs within the graph |
| `in2_paths`, `out2_paths` | number of 2-hop paths into / out of the transaction |
| `depth_up`, `depth_down` | longest chain of transactions leading into / out of it |
| `n_anc`, `n_desc` | number of transactions upstream / downstream (all reachable) |
| `chain_run` | length of a run of 1-in, 1–2-out transactions ending here (peel-chain-like pattern) |

In [5]:
t0 = time.time()
gf = pd.DataFrame(index=pd.Index(step.index, name='txId'))
in_deg = edges.txId2.value_counts()
out_deg = edges.txId1.value_counts()
gf['in_deg'] = in_deg.reindex(gf.index).fillna(0)
gf['out_deg'] = out_deg.reindex(gf.index).fillna(0)
gf['deg_ratio'] = (gf.out_deg + 1) / (gf.in_deg + 1)
gf['is_source'] = (gf.in_deg == 0).astype(int)
gf['is_sink'] = (gf.out_deg == 0).astype(int)
gf['in2_paths'] = edges.assign(d=edges.txId1.map(in_deg).fillna(0)).groupby('txId2').d.sum().reindex(gf.index).fillna(0)
gf['out2_paths'] = edges.assign(d=edges.txId2.map(out_deg).fillna(0)).groupby('txId1').d.sum().reindex(gf.index).fillna(0)

# longest path in / out and peel-chain runs via one pass in topological order
order = list(nx.topological_sort(G))
depth_up, chain = {}, {}
for v in order:
    preds = list(G.predecessors(v))
    depth_up[v] = max((depth_up[u] + 1 for u in preds), default=0)
    chain_like = len(preds) == 1 and 1 <= G.out_degree(v) <= 2
    chain[v] = chain[preds[0]] + 1 if chain_like else 0
depth_down = {}
for v in reversed(order):
    depth_down[v] = max((depth_down[w] + 1 for w in G.successors(v)), default=0)

gf['depth_up'] = gf.index.map(depth_up).fillna(0)
gf['depth_down'] = gf.index.map(depth_down).fillna(0)
gf['chain_run'] = gf.index.map(chain).fillna(0)
gf['n_anc'] = [len(nx.ancestors(G, n)) if n in G else 0 for n in gf.index]
gf['n_desc'] = [len(nx.descendants(G, n)) if n in G else 0 for n in gf.index]

struct_cols = list(gf.columns)
print(f'{len(struct_cols)} structural features in {time.time() - t0:.1f}s')
gf.describe().T.round(2)

12 structural features in 8.1s


,count,mean,std,min,25%,50%,75%,max
in_deg,157101.0,1.24,4.34,0.0,0.0,1.0,1.0,284.0
out_deg,157101.0,1.16,1.94,0.0,1.0,1.0,1.0,472.0
deg_ratio,157101.0,1.35,1.34,0.0,1.0,1.0,2.0,236.5
is_source,157101.0,0.27,0.44,0.0,0.0,0.0,1.0,1.0
is_sink,157101.0,0.18,0.39,0.0,0.0,0.0,0.0,1.0
in2_paths,157101.0,1.30,6.06,0.0,0.0,1.0,1.0,326.0
out2_paths,157101.0,1.24,4.29,0.0,0.0,1.0,1.0,478.0
depth_up,157101.0,26.36,91.45,0.0,0.0,2.0,11.0,1248.0
depth_down,157101.0,25.00,89.07,0.0,1.0,3.0,10.0,1248.0
chain_run,157101.0,20.10,88.04,0.0,0.0,0.0,3.0,1247.0


### 4b. Neighbour feature aggregates (visibility-masked)

For each transaction: mean and max of selected features over its **funding** (in) neighbours and its **spending** (out) neighbours, plus how many such neighbours were visible.
Only labelled-train and test transactions count as neighbours, so training rows see their neighbourhood the same way test rows do (Section 3).
Neighbour **labels** are never used: test neighbours have no labels, so that would not transfer.

To keep the feature count manageable, aggregates are built for the 20 raw features most important to a LightGBM model trained on F1's training steps only (1–20), so the selection never sees any validation step.

In [6]:
tr1 = lab[lab.time_step.isin(FOLDS['F1'][0])]
imp_model = LGBMClassifier(random_state=SEED, verbose=-1, importance_type='gain').fit(tr1[feat_cols], tr1['label'])
top_feats = pd.Series(imp_model.feature_importances_, index=feat_cols).nlargest(20).index.tolist()
print('aggregated features:', top_feats)

visible = pd.concat([lab[['txId'] + top_feats], test[['txId'] + top_feats]]).set_index('txId')
ve = edges[edges.txId1.isin(visible.index) & edges.txId2.isin(visible.index)]

in_agg = ve.join(visible, on='txId1').groupby('txId2')[top_feats].agg(['mean', 'max'])
in_agg.columns = [f'in_{f}_{s}' for f, s in in_agg.columns]
out_agg = ve.join(visible, on='txId2').groupby('txId1')[top_feats].agg(['mean', 'max'])
out_agg.columns = [f'out_{f}_{s}' for f, s in out_agg.columns]

nf = pd.DataFrame(index=gf.index)
nf['n_vis_in'] = ve.txId2.value_counts().reindex(nf.index).fillna(0)
nf['n_vis_out'] = ve.txId1.value_counts().reindex(nf.index).fillna(0)
nf = nf.join(in_agg).join(out_agg)           # NaN when no visible neighbour; LightGBM handles NaN natively
nbr_cols = list(nf.columns)
print(len(nbr_cols), 'neighbour features')

aggregated features: ['feat_90', 'feat_53', 'feat_47', 'feat_2', 'feat_163', 'feat_41', 'feat_132', 'feat_100', 'feat_16', 'feat_5', 'feat_80', 'feat_76', 'feat_49', 'feat_59', 'feat_8', 'feat_58', 'feat_138', 'feat_142', 'feat_119', 'feat_103']


82 neighbour features


In [7]:
lab_x = lab.join(gf, on='txId').join(nf, on='txId')
test_x = test.join(gf, on='txId').join(nf, on='txId')

FEATURE_SETS = {
    'base': feat_cols,
    'graph': feat_cols + struct_cols,
    'graph_nbr': feat_cols + struct_cols + nbr_cols,
}
{k: len(v) for k, v in FEATURE_SETS.items()}

{'base': 165, 'graph': 177, 'graph_nbr': 259}

## 5. Models

In [8]:
def make_model(name):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=500, max_features='sqrt', n_jobs=-1, random_state=SEED)
    if name == 'lgbm':
        return LGBMClassifier(random_state=SEED, verbose=-1)
    raise ValueError(name)

## 6. Experiments

| id | stage | model | features |
| --- | --- | --- | --- |
| exp01b | B3 comparison | Random Forest | base |
| exp01 | B3 winner (submitted) | LightGBM defaults | base |
| exp02 | D1 | LightGBM defaults | base + structural |
| exp03 | D1 | LightGBM defaults | base + structural + neighbour aggregates |

All LightGBM runs keep default parameters so differences come from the features alone.

In [9]:
EXPERIMENTS = [
    ('exp01b', 'rf', 'base'),
    ('exp01', 'lgbm', 'base'),
    ('exp02', 'lgbm', 'graph'),
    ('exp03', 'lgbm', 'graph_nbr'),
]

def run_cv(model_name, cols):
    res = {}
    for fold, (tr_steps, va_steps) in FOLDS.items():
        tr = lab_x[lab_x.time_step.isin(tr_steps)]
        va = lab_x[lab_x.time_step.isin(va_steps)]
        model = make_model(model_name).fit(tr[cols], tr['label'])
        res[fold] = roc_auc_score(va['label'], model.predict_proba(va[cols])[:, 1])
    return res

rows = []
for exp_id, m, fs in EXPERIMENTS:
    t0 = time.time()
    res = run_cv(m, FEATURE_SETS[fs])
    rows.append({'exp_id': exp_id, 'model': m, 'features': fs, **res, 'sec': round(time.time() - t0, 1)})
    print(exp_id, m, fs, {k: round(v, 4) for k, v in res.items()})

summary = pd.DataFrame(rows).set_index('exp_id')
summary['mean'] = summary[list(FOLDS)].mean(axis=1)
summary['min'] = summary[list(FOLDS)].min(axis=1)
summary = summary.sort_values(['F1', 'mean'], ascending=False)    # selection rule C1
summary.round(4)

exp01b rf base {'F1': 0.8781, 'F2': 0.9953, 'F3': 0.9953}


exp01 lgbm base {'F1': 0.9451, 'F2': 0.9964, 'F3': 0.9965}


exp02 lgbm graph {'F1': 0.9434, 'F2': 0.9961, 'F3': 0.9963}


exp03 lgbm graph_nbr {'F1': 0.9599, 'F2': 0.9968, 'F3': 0.996}


,model,features,F1,F2,F3,sec,mean,min
exp_id,,,,,,,,
exp03,lgbm,graph_nbr,0.9599,0.9968,0.9960,9.4,0.9842,0.9599
exp01,lgbm,base,0.9451,0.9964,0.9965,6.6,0.9793,0.9451
exp02,lgbm,graph,0.9434,0.9961,0.9963,6.0,0.9786,0.9434
exp01b,rf,base,0.8781,0.9953,0.9953,32.6,0.9562,0.8781


## 7. Fit the selected experiment on all labelled steps (1–35) and write its submission

Selection rule C1: highest F1 AUC, mean fold AUC as tie-break. Writing a file does not mean it gets submitted; that is decided after reviewing the table above.

In [10]:
best = summary.index[0]
best_model, best_fs = summary.loc[best, 'model'], summary.loc[best, 'features']
cols = FEATURE_SETS[best_fs]
print('Selected:', best, best_model, best_fs)

final = make_model(best_model).fit(lab_x[cols], lab_x['label'])
sub = sample[['index']].copy()
sub['target'] = final.predict_proba(test_x[cols])[:, 1]

# schema checks against the template
assert sub.shape == sample.shape and list(sub.columns) == list(sample.columns)
assert (sub['index'].values == sample['index'].values).all()
assert np.isfinite(sub['target']).all() and sub['target'].between(0, 1).all()

sub_path = SUB_DIR / f'{best}_{best_model}_{best_fs}.csv'
sub.to_csv(sub_path, index=False)
print('saved', sub_path)

Selected: exp03 lgbm graph_nbr


saved submissions\exp03_lgbm_graph_nbr.csv


In [11]:
# which features drive the selected model (gain importance)
imp = pd.Series(final.booster_.feature_importance('gain'), index=cols) if best_model == 'lgbm' \
      else pd.Series(final.feature_importances_, index=cols)
imp = (imp / imp.sum()).sort_values(ascending=False)
print('share of importance from structural features:', round(imp.reindex(struct_cols).sum(), 3) if best_fs != 'base' else 0)
print('share of importance from neighbour features:', round(imp.reindex(nbr_cols).sum(), 3) if best_fs == 'graph_nbr' else 0)
imp.head(20).round(4)

share of importance from structural features: 0.051
share of importance from neighbour features: 0.088


feat_53            0.2645
feat_90            0.1140
feat_55            0.0952
feat_5             0.0785
feat_163           0.0472
chain_run          0.0461
feat_2             0.0310
in_feat_59_mean    0.0224
feat_132           0.0160
out_feat_59_max    0.0155
feat_59            0.0137
feat_46            0.0136
feat_76            0.0131
in_feat_53_mean    0.0120
feat_80            0.0120
feat_138           0.0115
feat_22            0.0112
feat_103           0.0088
feat_16            0.0068
feat_41            0.0062
dtype: float64

## 8. Experiment log

In [12]:
prev = pd.read_csv(LOG) if LOG.exists() else pd.DataFrame(columns=['exp_id', 'public_lb'])
log = pd.DataFrame([{
    'exp_id': e, 'model': r['model'], 'features': r['features'], 'seed': SEED,
    'params': str(make_model(r['model']).get_params()),
    **{f'auc_{f}': round(r[f], 4) for f in FOLDS},
    'cv_mean': round(r['mean'], 4), 'cv_min': round(r['min'], 4),
} for e, r in summary.iterrows()])
log = log.merge(prev[['exp_id', 'public_lb']], on='exp_id', how='left')   # keep recorded public scores
log['submission'] = log.exp_id.map({best: str(sub_path)})
log = log.sort_values('exp_id')
log.to_csv(LOG, index=False)
log[['exp_id', 'model', 'features', 'auc_F1', 'auc_F2', 'auc_F3', 'cv_mean', 'public_lb', 'submission']]

,exp_id,model,features,auc_F1,auc_F2,auc_F3,cv_mean,public_lb,submission
1,exp01,lgbm,base,0.9451,0.9964,0.9965,0.9793,0.94664,NaN
3,exp01b,rf,base,0.8781,0.9953,0.9953,0.9562,NaN,NaN
2,exp02,lgbm,graph,0.9434,0.9961,0.9963,0.9786,NaN,NaN
0,exp03,lgbm,graph_nbr,0.9599,0.9968,0.9960,0.9842,NaN,submissions\exp03_lgbm_graph_nbr.csv
